# BioMarket - Store Management System

## Introduction

The project consists of developing a command-line management system for BioMarket s.a.s., a small shop selling vegan and organic products located at Via Tan 6.

The program runs from the command line and allows users to manage the main operations of the shop: adding products, viewing the inventory, recording sales, and calculating profits.

Particular attention has been given to data persistence, ensuring that information is not lost when the program is closed, and to input validation, so that invalid values can be handled without interrupting the execution of the program.

### Design Choices

A dictionary was used to represent the products, allowing each product name to be directly associated with its information: available quantity, purchase price, and selling price.

The management system data is stored in JSON format, a simple and readable solution that allows inventory and profit information to be preserved between different program sessions.

The different features are organized into dedicated functions in order to keep the code clear, modular, and easily reusable.

## Initial Setup

In this section, the required modules are imported and the file used for data persistence is defined.

In [10]:
import json

DATA_FILE = "biomarket_data.json"

## Data Persistence

To ensure data persistence, inventory and profit data are stored in the `biomarket_data.json` JSON file.

The `save_data()` function saves the current state of the management system, while `load_data()` retrieves the data when the program starts. If the file does not exist or contains invalid data, an empty data structure is automatically initialized.

In [11]:
def save_data(data):
    """Save the application data to a JSON file."""
    with open(DATA_FILE, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


def load_data():
    """Load application data from the JSON file."""
    try:
        with open(DATA_FILE, "r", encoding="utf-8") as file:
            return json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        return {
            "products": {},
            "gross_profit": 0.0,
            "net_profit": 0.0,
        }


data = load_data()

## Input Validation

To make the program more robust, user inputs are checked through dedicated validation functions.

Quantities must be positive integers, while prices must be positive numeric values. Empty text fields and answers to `yes/no` questions are also validated.

Exception handling makes it possible to detect invalid inputs, display a clear error message, and ask the user to enter the value again without interrupting the execution of the program.

In [12]:
def get_positive_integer(prompt):
    """Read and return a positive integer from user input."""
    while True:
        try:
            value = int(input(prompt))

            if value <= 0:
                raise ValueError

            return value

        except ValueError:
            print("Invalid value. Please enter a positive integer.")


def get_positive_float(prompt):
    """Read and return a positive floating-point number from user input."""
    while True:
        try:
            value = float(input(prompt))

            if value <= 0:
                raise ValueError

            return value

        except ValueError:
            print("Invalid value. Please enter a positive number.")


def get_non_empty_string(prompt):
    """Read and return a non-empty string from user input."""
    while True:
        value = input(prompt).strip()

        if value:
            return value

        print("Invalid value. Please enter some text.")


def get_yes_no(prompt):
    """Read and return a valid yes/no answer from user input."""
    while True:
        answer = input(prompt).strip().lower()

        if answer in ("yes", "no"):
            return answer

        print("Invalid answer. Please enter yes or no.")

## Help Command

The `help` command displays the complete list of operations available in the management system, along with a brief description of their purpose.

The same help screen is also displayed when the user enters an unrecognized command, making the program easier to use.

In [13]:
def show_help():
    """Display the list of available commands."""
    print("The available commands are:")
    print("add: add a product to the inventory")
    print("list: list the products in the inventory")
    print("sale: record a completed sale")
    print("profits: display total profits")
    print("help: display the available commands")
    print("close: exit the program")


def show_invalid_command():
    """Display an invalid command message and the help menu."""
    print("Invalid command")
    show_help()

## Add Command

The `add` command allows the user to register a new product by entering its name, quantity, purchase price, and selling price.

If the product is already available in the inventory, the program only updates the available quantity without asking for the previously stored prices again.

After each modification, the data is saved to the JSON file to keep the inventory information up to date.

In [14]:
def add_product(data):
    """Add a new product or update an existing product."""
    product_name = get_non_empty_string("Product name: ").lower()

    if product_name in data["products"]:
        quantity = get_positive_integer("Quantity: ")
        data["products"][product_name]["quantity"] += quantity
        save_data(data)
        print(f"ADDED: {quantity} X {product_name}")
        return

    quantity = get_positive_integer("Quantity: ")
    purchase_price = get_positive_float("Purchase price: ")
    sale_price = get_positive_float("Selling price: ")

    data["products"][product_name] = {
        "quantity": quantity,
        "purchase_price": purchase_price,
        "sale_price": sale_price,
    }

    save_data(data)
    print(f"ADDED: {quantity} X {product_name}")

## List Command

The `list` command displays all products currently available in the inventory, showing the name, available quantity, and selling price for each product.

Products with a quantity equal to zero are not displayed, so that only items that are actually available are shown.

In [15]:
def list_products(data):
    """Display all products currently available in stock."""
    print("PRODUCT\tQUANTITY\tPRICE")

    products_found = False

    for product_name, product_info in data["products"].items():
        if product_info["quantity"] > 0:
            products_found = True
            print(
                f"{product_name}\t"
                f"{product_info['quantity']}\t"
                f"€{product_info['sale_price']}"
            )

    if not products_found:
        print("No products in stock.")

## Sale Command

The `sale` command allows the user to record a transaction containing one or more products.

Before recording each item, the program checks that the product exists in the inventory and that the requested quantity is available. If not, an error message is displayed without interrupting the program.

For each product sold, the stock quantity is updated, and both the total sale amount and the corresponding purchase cost are calculated. At the end of the transaction, the data is saved and the overall profits are updated.

In [16]:
def register_sale(data):
    """Register a sale and update stock quantities."""
    sale_items = []
    reserved_quantities = {}
    sale_total = 0.0
    sale_cost = 0.0

    while True:
        product_name = get_non_empty_string("Product name: ").lower()

        if product_name not in data["products"]:
            print("Product not found in inventory.")
            retry = get_yes_no("Would you like to try again? (yes/no): ")

            if retry == "no":
                print("Sale cancelled")
                return

            continue

        available_quantity = (
            data["products"][product_name]["quantity"]
            - reserved_quantities.get(product_name, 0)
        )

        if available_quantity == 0:
            print("Product out of stock.")
            retry = get_yes_no("Would you like to try again? (yes/no): ")

            if retry == "no":
                print("Sale cancelled")
                return

            continue

        quantity = get_positive_integer("Quantity: ")

        if quantity > available_quantity:
            print("Requested quantity is not available in inventory.")
            retry = get_yes_no("Would you like to try again? (yes/no): ")

            if retry == "no":
                print("Sale cancelled")
                return

            continue

        sale_price = data["products"][product_name]["sale_price"]
        purchase_price = data["products"][product_name]["purchase_price"]

        item_total = quantity * sale_price
        item_cost = quantity * purchase_price

        sale_items.append(
            {
                "name": product_name,
                "quantity": quantity,
                "sale_price": sale_price,
            }
        )

        reserved_quantities[product_name] = (
            reserved_quantities.get(product_name, 0) + quantity
        )

        sale_total += item_total
        sale_cost += item_cost

        another_product = get_yes_no(
            "Would you like to add another product? (yes/no): "
        )

        if another_product == "no":
            break

    for item in sale_items:
        data["products"][item["name"]]["quantity"] -= item["quantity"]

    data["gross_profit"] += sale_total
    data["net_profit"] += sale_total - sale_cost

    save_data(data)

    print("SALE RECORDED")

    for item in sale_items:
        prefix = "- " if len(sale_items) > 1 else ""
        print(
            f"{prefix}{item['quantity']} X {item['name']}: "
            f"€{item['sale_price']:.2f}"
        )

    print(f"Total: €{sale_total:.2f}")

## Profits Command

The `profits` command displays the gross profit and net profit accumulated from the recorded sales.

Gross profit corresponds to the total revenue generated from sales, while net profit is calculated by subtracting the purchase cost of the sold products from the total sales revenue.

In [17]:
def show_profits(data):
    """Display gross and net profits."""
    gross_profit = data["gross_profit"]
    net_profit = data["net_profit"]

    print(
        f"Profit: gross=€{gross_profit:.2f} "
        f"net=€{net_profit:.2f}"
    )

## Main Program

The `main()` function manages the main program loop and processes the commands entered by the user.

Based on the command received, the corresponding function is called. The loop continues until the user enters `close`, while unrecognized commands are handled by displaying an error message and the help menu.

In [9]:
def main(data):
    """Run the main command loop of the application."""
    while True:
        command = input("Enter a command: ").strip().lower()

        if command == "help":
            show_help()

        elif command == "add":
            add_product(data)

        elif command == "list":
            list_products(data)

        elif command == "sale":
            register_sale(data)

        elif command == "profits":
            show_profits(data)

        elif command == "close":
            print("Bye bye")
            break

        else:
            show_invalid_command()

        print()


if __name__ == "__main__":
    main(data)

Enter a command: help
The available commands are:
add: add a product to the inventory
list: list the products in the inventory
sale: record a completed sale
profits: display total profits
help: display the available commands
close: exit the program

Enter a command: add
Product name: soy milk
Quantity: 20
Purchase price: 0.80
Selling price: 1.40
ADDED: 20 X soy milk

Enter a command: add
Product name: tofu
Quantity: 10
Purchase price: 2.20
Selling price: 4.19
ADDED: 10 X tofu

Enter a command: add
Product name: seitan
Quantity: 5
Purchase price: 3
Selling price: 5.49
ADDED: 5 X seitan

Enter a command: list
PRODUCT	QUANTITY	PRICE
soy milk	20	€1.4
tofu	10	€4.19
seitan	5	€5.49

Enter a command: sale
Product name: soy milk
Quantity: 5
Would you like to add another product? (yes/no): yes
Product name: tofu
Quantity: 2
Would you like to add another product? (yes/no): no
SALE RECORDED
- 5 X soy milk: €1.40
- 2 X tofu: €4.19
Total: €15.38

Enter a command: list
PRODUCT	QUANTITY	PRICE
soy milk

## Conclusions

The project resulted in the development of a text-based Python management system capable of supporting BioMarket's main operations: inventory management, sales recording, and profit calculation.

The solution was developed using a modular structure, dividing the different features into dedicated functions and using a dictionary to organize product information in a structured way. Data persistence through a JSON file also allows inventory and profit data to be preserved between different executions of the program.

Particular attention was given to the quality and reliability of the entered data. Input validation and exception handling make it possible to detect invalid values and reduce the risk of errors during the use of the management system.

Overall, the project represents a simple application of the data management cycle: input collection, validation, organization, updating, persistence, and use of information to obtain indicators such as gross profit and net profit.

The adopted structure also provides an easily extendable foundation for future developments, such as sales history analysis or the creation of additional indicators to evaluate the store's performance.